# Sneak integration receipt — SNEAK-014

Observed: hosted Cognee ingested and retrieved one synthetic record; bounded graph returned 12 nodes and 16 edges; a supervised bridge changed Jerry's answer from UNKNOWN to a source-cited proposal. Flexprice SDK installed. Snyk SCA found no known vulnerable paths in 12 integration dependencies; SAST reported four LOW findings.

This notebook verifies captured records offline. It does not replay provider writes or claim autonomous tools, mandatory Cognee OSS acceptance, a running source-of-truth gateway, zero application findings, or an awarded score. No credentials are embedded. Source code scanned: commit `1199675accec5294d056de16b2538b58850f3bc3`.


In [1]:
from pathlib import Path
import json, hashlib, subprocess
ROOT=Path.cwd()
if not (ROOT/'receipts/integrations/SNEAK-014_EVIDENCE.json').exists(): ROOT=ROOT.parent
r=json.loads((ROOT/'receipts/integrations/SNEAK-014_EVIDENCE.json').read_text())
events={e['id']:e['payload'] for e in r['events']}
cog=events['R14-COGNEE']; source=cog['ingest']['MEMORY_SOURCE']
retrieved=cog['recall']['RECALL'][0]['text']
assert hashlib.sha256(retrieved.encode()).hexdigest()==source['identity']['content_sha256']
assert cog['recall']['GRAPH']['nodes']==12 and cog['recall']['GRAPH']['edges']==16
messages=events['R14-BAND']['messages']
assert any('UNKNOWN' in m['content'] for m in messages)
assert any(source['memory']['review_label'] in m['content'] and 'QUARANTINE_RESCAN' in m['content'] for m in messages)
assert len(events['R14-SNYK']['sast']['findings'])==4
assert events['R14-SNYK']['sca']['dependency_count']==12
assert events['R14-SNYK']['sca']['known_vulnerabilities']==0
print({'source_bytes_match':True,'bounded_graph_nodes':12,'bounded_graph_edges':16,'memory_transport':'supervised','sast_findings':4,'sca_known_vulnerabilities':0,'flexprice_sdk':'2.1.40','live_billing':'NOT_EXECUTED'})


{'source_bytes_match': True, 'bounded_graph_nodes': 12, 'bounded_graph_edges': 16, 'memory_transport': 'supervised', 'sast_findings': 4, 'sca_known_vulnerabilities': 0, 'flexprice_sdk': '2.1.40', 'live_billing': 'NOT_EXECUTED'}


In [2]:
def h(b): return hashlib.sha256(b).digest()
def canonical(x): return json.dumps(x,sort_keys=True,ensure_ascii=True,separators=(',',':')).encode('ascii')
def parent(a,b): return h(b'\x01'+a+b)
def tree(xs):
    levels=[xs]
    while len(levels[-1])>1:
        row=levels[-1];levels.append([parent(row[i],row[i+1]) if i+1<len(row) else row[i] for i in range(0,len(row),2)])
    return levels
ordered=sorted(r['events'],key=lambda e:e['id']); payloads=[canonical(e) for e in ordered]; leaves=[h(b'\x00'+b) for b in payloads]
levels=tree(leaves);root=levels[-1][0]
proofs=[]
for index in range(len(leaves)):
    proof=[]
    for row in levels[:-1]:
        if index%2: proof.append({'side':'left','hash':row[index-1].hex()})
        elif index+1<len(row): proof.append({'side':'right','hash':row[index+1].hex()})
        index//=2
    proofs.append(proof)
receipt={'schema':'sneak.integration-event-merkle.v1','id':'SNEAK-014-CUSTODY','parent':'SNEAK-014','parent_root':'29002d5e05b18b58dceec7d1492a1300efc9bbf7d33e0fdafed755f43561895f','scope':'Five captured integration event records, not the full repository or vendor systems','construction':{'hash':'SHA-256','canonical':'ASCII JSON recursively sorted keys ensure_ascii=True compact separators no newline','ordering':'event id ascending','leaf':'SHA256(0x00 || canonical(event))','internal':'SHA256(0x01 || left32 || right32)','odd':'promote unchanged'},'root':root.hex(),'leaf_count':len(leaves),'leaves':[{'event':e,'canonical_hex':b.hex(),'leaf':x.hex(),'proof':p} for e,b,x,p in zip(ordered,payloads,leaves,proofs)],'mmr':'NOT_COMMITTED','signed':False,'meaning':'Integrity of captured evidence, not truth, causality, permissions, provider attestation or absence of vulnerabilities'}
assert tree(leaves[:-1])[-1][0]!=root
print({'python_root':root.hex(),'leaves':len(leaves),'removed_leaf_preserves_root':False})


{'python_root': '89d9c17ace42bd7fbbbc20eb42c572161cc6e9f95ead4202faf63057a5db9276', 'leaves': 5, 'removed_leaf_preserves_root': False}


In [3]:
NODE="const fs=require(\"node:fs\"),crypto=require(\"node:crypto\"),assert=require(\"node:assert/strict\");\nconst r=JSON.parse(fs.readFileSync(0,\"utf8\"));\nconst sort=x=>Array.isArray(x)?x.map(sort):x&&typeof x===\"object\"?Object.fromEntries(Object.keys(x).sort().map(k=>[k,sort(x[k])])):x;\nconst canon=x=>JSON.stringify(sort(x)).replace(/[\\u007f-\\uffff]/g,c=>\"\\\\u\"+c.charCodeAt(0).toString(16).padStart(4,\"0\"));\nconst h=b=>crypto.createHash(\"sha256\").update(b).digest();\nconst par=(a,b)=>h(Buffer.concat([Buffer.from([1]),a,b]));\nconst ids=r.leaves.map(x=>x.event.id);assert.deepEqual(ids,[...ids].sort());assert.equal(r.leaf_count,ids.length);\nlet xs=r.leaves.map(item=>{\n const b=Buffer.from(canon(item.event),\"ascii\");assert.equal(b.toString(\"hex\"),item.canonical_hex);\n const leaf=h(Buffer.concat([Buffer.from([0]),b]));assert.equal(leaf.toString(\"hex\"),item.leaf);\n let v=leaf;\n for(const p of item.proof){assert.ok([\"left\",\"right\"].includes(p.side));assert.match(p.hash,/^[0-9a-f]{64}$/);const s=Buffer.from(p.hash,\"hex\");v=p.side===\"left\"?par(s,v):par(v,s);}\n assert.equal(v.toString(\"hex\"),r.root);return leaf;\n});\nwhile(xs.length>1){const n=[];for(let i=0;i<xs.length;i+=2)n.push(i+1<xs.length?par(xs[i],xs[i+1]):xs[i]);xs=n;}\nassert.equal(xs[0].toString(\"hex\"),r.root);\nconsole.log(JSON.stringify({independent_node:\"PASS\",root:r.root,leaves:r.leaf_count}));"
assert isinstance(NODE, str) and 'createHash' in NODE
checked=subprocess.run(['node','-e',NODE],input=json.dumps(receipt),text=True,capture_output=True)
assert checked.returncode==0,checked.stderr
print(checked.stdout)
out=ROOT/'receipts/integrations/SNEAK-014_CUSTODY.json'
if out.exists(): assert json.loads(out.read_text())==receipt, 'Existing receipt differs: create a successor'
else: out.write_text(json.dumps(receipt,indent=2)+'\n')


{"independent_node":"PASS","root":"89d9c17ace42bd7fbbbc20eb42c572161cc6e9f95ead4202faf63057a5db9276","leaves":5}



## Live verification and limits

For a read-only live recall, export `COGNEE_BASE_URL` and `COGNEE_API_KEY` privately, import `CogneeClient` from `scripts/integrations_bridge.py`, and query only `sneak_custody_r14`. Compare the returned text hash with the captured record. Provider state can change; preserve a successor on differences.

This notebook does not ingest live data by default. The synthetic fixture is not a real vulnerability. Snyk's four LOW findings are retained separately and need review/remediation. Flexprice needs a configured test credential and real event readback. Mandatory Cognee OSS, agent-owned MCP memory tools, BAND Desktop visibility, sandboxed LOMO and gateway enforcement remain NOT_TESTED or NOT_IMPLEMENTED. Captured human choice remains PENDING; no approval is inferred from provisioning credentials.
